In [ ]:
best_umap_model = UMAP(
    n_neighbors=best_params["n_neighbors"],
    n_components=best_params["n_components"],
    min_dist=0.0,
    metric="cosine",
    random_state=RANDOM_STATE,
    low_memory=True
)

best_hdbscan_model = HDBSCAN(
    min_cluster_size=best_params["min_cluster_size"],
    min_samples=best_params["min_samples"],
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True,
    core_dist_n_jobs=-1
)

best_vectorizer_model = CountVectorizer(
    min_df=2,
    max_df=0.8,
    #ngram_range=(1, 1)
)

best_ctfidf_model = ClassTfidfTransformer(
    reduce_frequent_words=True,
    bm25_weighting=True
)

best_topic_model = BERTopic(
    language="french",
    hdbscan_model=best_hdbscan_model,
    umap_model=best_umap_model,
    vectorizer_model=best_vectorizer_model,
    ctfidf_model=best_ctfidf_model,
    calculate_probabilities=False,
    nr_topics=None,
    verbose=True
)


best_topics_raw, _ = best_topic_model.fit_transform(
    documents,
    embeddings=embeddings_array
)

best_topics_raw = np.asarray(best_topics_raw)


nombre_topics_bruts = len(
    np.unique(
        best_topics_raw[
            best_topics_raw != -1
        ]
    )
)

taux_outliers_brut = np.mean(best_topics_raw == -1)

print("Nombre naturel de topics :", nombre_topics_bruts)

print( f"Taux brut d'outliers : " f"{taux_outliers_brut:.1%}")

display(best_topic_model.get_topic_info())

In [ ]:
seuils_outliers = [
    0.0,
    0.10,
    0.20,
    0.30,
    0.40,
    0.50
]

comparaisons_seuils = []
topics_par_seuil = {}

for seuil in seuils_outliers:
    topics_test = best_topic_model.reduce_outliers(
        documents,
        best_topics_raw,
        strategy="embeddings",
        embeddings=embeddings_array,
        threshold=seuil)

    topics_test = np.asarray(topics_test)

    topics_par_seuil[seuil] = topics_test

    masque_assignes = topics_test != -1

    tailles_topics = (pd.Series(topics_test[masque_assignes]).value_counts())

    nombre_assignes = int(masque_assignes.sum())

    nombre_reassignes = int(((best_topics_raw == -1) & (topics_test != -1)).sum())

    comparaisons_seuils.append({
        "threshold": seuil,
        "n_topics": len(tailles_topics),
        "outlier_rate_remaining": np.mean(
            topics_test == -1
        ),
        "n_reassigned": nombre_reassignes,
        "smallest_topic": (
            tailles_topics.min()
            if len(tailles_topics) > 0
            else np.nan
        ),
        "largest_topic": (
            tailles_topics.max()
            if len(tailles_topics) > 0
            else np.nan
        ),
        "largest_topic_share": (
            tailles_topics.max() / nombre_assignes
            if nombre_assignes > 0
            else np.nan
        )
    })


comparaison_seuils_df = pd.DataFrame(comparaisons_seuils)

display(
    comparaison_seuils_df.style.format({
        "threshold": "{:.2f}",
        "outlier_rate_remaining": "{:.1%}",
        "largest_topic_share": "{:.1%}"
    })
)

In [ ]:
SEUIL_OUTLIERS_FINAL = 0.30

topics_finaux = np.asarray(topics_par_seuil[SEUIL_OUTLIERS_FINAL])

print(
    f"Taux d'outliers final : "
    f"{np.mean(topics_finaux == -1):.1%}"
)


print(
    "Nombre final de topics :",
    len(
        np.unique(
            topics_finaux[
                topics_finaux != -1
            ]
        )
    )
)

distribution_topics = (
    pd.Series(topics_finaux)
    .value_counts()
    .sort_index()
    .rename_axis("Topic")
    .reset_index(name="Count")
)

display(distribution_topics)

In [ ]:
stopwords_corpus = ["deberl", "men", "yole","embrass", "revien"]

vectorizer_final = CountVectorizer(
    stop_words=stopwords_corpus,
    min_df=2,
    max_df=0.80,
    #ngram_range=(1, 2)
)

ctfidf_final = ClassTfidfTransformer(
    reduce_frequent_words=True,
    bm25_weighting=True
)

best_topic_model.update_topics(
    documents,
    topics=topics_finaux,
    vectorizer_model=vectorizer_final,
    ctfidf_model=ctfidf_final,
    top_n_words=10
)

display(best_topic_model.get_topic_info())

In [ ]:
# L'analyseur produit exactement les tokens 
# attendus par le nouveau CountVectorizer.

analyseur_final = (vectorizer_final.build_analyzer())

texts_tokenises_final = [
    analyseur_final(document)
    for document in documents
]


dictionary_final = Dictionary(texts_tokenises_final)


coherence_finale = calculer_coherence_cv(
    topic_model=best_topic_model,
    labels=topics_finaux,
    texts_tokenises=texts_tokenises_final,
    dictionary=dictionary_final,
    top_n_words=10
)


diversite_finale = calculer_diversite_topics(
    topic_model=best_topic_model,
    labels=topics_finaux,
    top_n_words=10
)


print(
    f"Cohérence C_v finale : "
    f"{coherence_finale:.3f}"
)

print(
    f"Diversité finale : "
    f"{diversite_finale:.3f}"
)

In [ ]:
fig = best_topic_model.visualize_hierarchy()
fig.show()

In [ ]:
timestamps = df_model["annee"].tolist()

topics_over_time = best_topic_model.topics_over_time(
    documents,
    timestamps,
    nr_bins=15
)
best_topic_model.visualize_topics_over_time(topics_over_time) #topics=themes_interet)

In [ ]:
df_resultats = df_model.copy()

df_resultats["topic"] = topics_finaux

df_resultats["est_outlier"] = (df_resultats["topic"] == -1)

display(df_resultats[[COL_ROMAN, COL_TEXTE, "topic", "est_outlier"]].head())

In [ ]:
table_topics_romans_counts = pd.crosstab(df_resultats[COL_ROMAN],df_resultats["topic"])

display(table_topics_romans_counts)

In [ ]:
table_topics_romans_proportions = pd.crosstab(
    df_resultats[COL_ROMAN],
    df_resultats["topic"],
    normalize="index"
)

display(table_topics_romans_proportions.style.format("{:.1%}"))

In [ ]:
topics_par_roman = (
    best_topic_model
    .topics_per_class(
        documents,
        classes=romans,
        global_tuning=True
    )
)

display(topics_par_roman)

In [ ]:
resultats_df.to_csv("resultats_grille_bertopic.csv",index=False)



df_resultats.to_csv("documents_avec_topics.csv",index=False)

table_topics_romans_counts.to_csv("topics_par_roman_comptages.csv")

table_topics_romans_proportions.to_csv("topics_par_roman_proportions.csv")

topics_par_roman.to_csv("bertopic_topics_per_class.csv",index=False)

best_topic_model.get_topic_info().to_csv("table_topics_romans_counts.csv",index=False)